# Capstone G5: Predicting heart disease from a checkup

Thirteen numbers from a routine cardiac workup -- cholesterol, blood pressure, age, chest-pain type, an exercise test -- to flag heart disease. Classic clinical prediction on 303 patients.

**Your group's priority: fairness across sex.** Heart disease presents differently in women and is historically under-diagnosed in them. A model trained on mostly-male data can be accurate overall yet worse for women.

This is a **tabular** problem -- a table of numbers per person, like Day 2. The model is easy;
the interesting questions are about *which features* and *who the model works for*. **Rubric:**
build it, evaluate honestly, find a failure mode, defend a decision, both partners can explain.

In [ ]:
# Setup: on Colab this grabs the course files + installs what's missing. Locally it's a no-op.
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g5_heart")
sys.path.insert(0, "../..")            # day3_capstone (capstone_common / _tabular / _seq)
sys.path.insert(0, "../../../_shared") # colab_setup
import colab_setup
colab_setup.ensure(*colab_setup.DAY3_TABULAR)

In [ ]:
import sys
sys.path.insert(0, "../..")
import capstone_tabular as ct

df, meta = ct.load_heart()
print(meta["name"], "  ->", df.shape[0], "rows")
print("features:", meta["features"])
print("predicting:", meta["target"], f"(1 = {meta['positive']})")
df.head()

## Look at the data
How many people are in each class? Is it balanced? What might be confounded?

In [ ]:
import matplotlib.pyplot as plt
print(df[meta["target"]].value_counts().rename({0: "no", 1: "yes"}).to_string())
df[meta["features"]].describe().T[["mean", "min", "max"]]

## Build your own model (interactive)

Pick which **features** the model sees and which **model** to use, hit Run, read the TEST
accuracy. Try dropping features: does accuracy hold with fewer? Which features actually matter?

In [ ]:
from ipywidgets import interact_manual, SelectMultiple, Dropdown

def build(features, model):
    if not features:
        print("Pick at least one feature."); return
    ct.fit_eval(df, list(features), meta["target"], model=model)

try:
    interact_manual(build,
        features=SelectMultiple(options=meta["features"], value=tuple(meta["features"]),
                                description="features", rows=min(10, len(meta["features"])),
                                style={"description_width": "initial"}),
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.fit_eval(df, meta["features"], meta["target"])

## Fairness: does the model work equally well for everyone?

Your priority is **fairness across sex**. Train once, then measure TEST accuracy **separately for
each group** (women vs men). A model can look great overall and quietly fail one group.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def fairness(model):
    ct.audit_by_group(df, meta["features"], meta["target"],
                      meta["group"], meta.get("group_names"), model=model)

try:
    interact_manual(fairness,
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.audit_by_group(df, meta["features"], meta["target"], meta["group"], meta.get("group_names"))

## Where to take it (with Claude)
- Does **cholesterol** alone predict much? Compare a chol-only model to the full one -- you may be surprised which features carry the signal.
- The dataset is ~2:1 male. Does that explain the accuracy gap by sex? What would you collect to fix it?
- Turn it into a risk *score* (probability) instead of yes/no -- how would a doctor use that?